# **Test d'intégration du loader/Module/InfoNCELoss pour la phase patient-wise-learning**

In [40]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

In [69]:
%load_ext autoreload
%autoreload 2

from gbmhackathon.training.patientwise import *
from gbmhackathon.models.mme import MultiModalEncoder, AuxilliaryClassifier
from gbmhackathon.utils.loss_functions import InfoNCELoss, RegularizedInfoNCELoss, SmoothingFunction, RankMe
from gbmhackathon.s3_loader import load_s3

import os
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# To investigate gradients
from torchviz import make_dot

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
torch.set_num_threads(6)
torch.get_num_threads()

6

In [71]:
device = "cpu" #"cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

In [72]:
name_emb_dict = {"hne":"embeddings_HnE_OptimusH0.pkl",
#"spatial":"2025-03-23_18-32_spatial_emb_V1.pkl",
"clinical":"2025-03-30_14-23_clinical_emb_V1.pkl",
"wes":"2025-04-05_13-40_wes_emb_V1.pkl",
"bulk":"2025-05-03_10-15_bulk_emb_V1.pkl",
"scRNA":"2025-05-04_02-35_scRNA_emb_V1.pkl"}
pkl_storage_folder = "embedding_V1"

In [99]:
dataset = PatientLearningDataset(name_emb_dict, pkl_storage_folder, device=device, dropout=0.35)
print(f"Dataset size: {len(dataset)}")
BATCH_SIZE = 128
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_patient_wise, generator=torch.Generator(device=dataset.device))

Using device: cpu
By keeping 35.00% of dropout augmented samples we went from:
468 dropout samples (80.41% dropout in dataset) -- to --> 167 dropout samples (59.43% dropout in dataset)
Dataset size: 281


In [101]:
batch_all = [dataset.__getitem__(idx) for idx in dataset.ind2patient]
batch_all = collate_patient_wise(batch_all)

## Pour la loss, il nous faut un dictionnaire qui associe chaque patient à un ID
On fait bien attention à indiquer les rechutes comme le même patient

In [102]:
missing_mods = load_s3("s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/missing_mod_per_samples.pkl")

In [103]:
all_ids = list(missing_mods.keys())
patient_map = {key: i for key, i in zip(all_ids, [k for k in range(1,len(all_ids)+1)])}
for patient_id, idx in patient_map.items():
    if patient_id.endswith('b'): # pour les rechutes on met le même index que le sample original
        patient_map[patient_id] = idx - 1

In [104]:
len(patient_map)

114

In [105]:
patient_number = []
for pid in dataset.ind2patient.values():
    if 'd' in pid:
        pid = pid[:pid.index('_d')]
    patient_number.append(patient_map[pid])
patient_map = dict(zip(dataset.ind2patient.values(), patient_number))

In [106]:
idx_compatible_mapping = dict(zip(list(set(patient_map.values())), [k for k in range(len(set(patient_map.values())))]))
for pid in patient_map.keys():
    patient_map[pid] = idx_compatible_mapping[patient_map[pid]]
patient_map

{'HK_G_068a': 61,
 'HK_G_016a': 15,
 'HK_G_039a': 35,
 'HK_G_079b': 71,
 'HK_G_072a': 65,
 'HK_G_081a': 73,
 'HK_G_084b': 74,
 'HK_G_003a': 2,
 'HK_G_032a': 29,
 'HK_G_064a': 57,
 'HK_G_082b': 73,
 'HK_G_086b': 75,
 'HK_G_099a': 84,
 'HK_G_045a': 40,
 'HK_G_018a': 16,
 'HK_G_006a': 5,
 'HK_G_092b': 79,
 'HK_G_054a': 49,
 'HK_G_098b': 83,
 'HK_G_014a': 13,
 'HK_G_070a': 63,
 'HK_G_056a': 51,
 'HK_G_004a': 3,
 'HK_G_060a': 54,
 'HK_G_035a': 32,
 'HK_G_063a': 56,
 'HK_G_011a': 10,
 'HK_G_005a': 4,
 'HK_G_022a': 20,
 'HK_G_051a': 46,
 'HK_G_044b': 39,
 'HK_G_029b': 26,
 'HK_G_036b': 32,
 'HK_G_114a': 94,
 'HK_G_093a': 80,
 'HK_G_111b': 92,
 'HK_G_024a': 22,
 'HK_G_008a': 7,
 'HK_G_108a': 91,
 'HK_G_069a': 62,
 'HK_G_030a': 27,
 'HK_G_019a': 17,
 'HK_G_027a': 25,
 'HK_G_095a': 82,
 'HK_G_053a': 48,
 'HK_G_112a': 93,
 'HK_G_028a': 26,
 'HK_G_104a': 88,
 'HK_G_090b': 78,
 'HK_G_020a': 18,
 'HK_G_077a': 70,
 'HK_G_050a': 45,
 'HK_G_073a': 66,
 'HK_G_085a': 75,
 'HK_G_034a': 31,
 'HK_G_010a': 9

## Voir à quoi ressemble l'output du loader

In [107]:
toy_batch = next(iter(dataloader))
toy_batch

(['HK_G_105b_dclinical',
  'HK_G_009a',
  'HK_G_068a_dbulk',
  'HK_G_056a_dclinical_bulk',
  'HK_G_008a',
  'HK_G_014a',
  'HK_G_068a',
  'HK_G_113b_dbulk_scRNA',
  'HK_G_028a',
  'HK_G_100b_dbulk',
  'HK_G_112a',
  'HK_G_107a_dclinical_wes',
  'HK_G_064a_dclinical_wes_scRNA',
  'HK_G_099a_dclinical_bulk',
  'HK_G_086b_dbulk_scRNA',
  'HK_G_053a',
  'HK_G_069a_dscRNA',
  'HK_G_086b_dclinical',
  'HK_G_109b',
  'HK_G_104a_dhne_bulk',
  'HK_G_094a_dwes_scRNA',
  'HK_G_110a_dbulk',
  'HK_G_073a_dclinical_wes',
  'HK_G_068a_dwes_scRNA',
  'HK_G_079b_dscRNA',
  'HK_G_094a',
  'HK_G_074a_dclinical',
  'HK_G_106a_dhne_clinical',
  'HK_G_059b',
  'HK_G_060a',
  'HK_G_013a',
  'HK_G_085a_dclinical',
  'HK_G_081a_dwes',
  'HK_G_099a_dwes_bulk_scRNA',
  'HK_G_081a_dbulk_scRNA',
  'HK_G_063a',
  'HK_G_073a',
  'HK_G_076a_dclinical_bulk_scRNA',
  'HK_G_064a_dwes_scRNA',
  'HK_G_099a_dwes_scRNA',
  'HK_G_064a_dwes',
  'HK_G_061b_dclinical_bulk_scRNA',
  'HK_G_078a_dbulk',
  'HK_G_095a_dhne',
  'HK_G

In [108]:
raw_emb = toy_batch[2]
inpute_size_dict = {}
for mod in raw_emb.keys():
    print(mod, raw_emb[mod].size())
    inpute_size_dict[mod] = raw_emb[mod].size(1)

hne torch.Size([128, 1536])
clinical torch.Size([128, 12])
wes torch.Size([128, 1790])
bulk torch.Size([128, 3072])
scRNA torch.Size([128, 3072])


# Pour instantier le MultiModalEncoder
Il faut des config pour chaque modalité, par simplicité on va définir une coquille de base et simplement ajouter la bonne dimension en entrée

In [109]:
out = 64
norm_fn = nn.LayerNorm
base_config = {"layers": [1024,512,out],
        "dropout": 0.65,
        "act_fn":SmoothingFunction,
        "norm_layer":norm_fn,
        "device":device}

high_capacity_cfg = {"layers": [256,512,256,out],
        "dropout": 0.65,
        "act_fn":SmoothingFunction,
        "norm_layer":norm_fn,
        "device":device}

mid_capacity_cfg = {"layers": [256,128,64,out],
        "dropout": 0.55,
        "act_fn":SmoothingFunction,
        "norm_layer":norm_fn,
        "device":device}

small_capacity_cfg = {"layers": [128,64,out],
        "dropout": 0.45,
        "act_fn":SmoothingFunction,
        "norm_layer":norm_fn,
        "device":device}

def adapt_base_config(base_cfg, input_size):
    base_copy = deepcopy(base_cfg)
    base_copy["layers"] = [input_size] + base_copy["layers"]
    return base_copy

hne_cfg = adapt_base_config(high_capacity_cfg, inpute_size_dict["hne"])
#spatial_cfg = adapt_base_config(small_capacity_cfg, inpute_size_dict["spatial"])
clinical_cfg = adapt_base_config(high_capacity_cfg, inpute_size_dict["clinical"])
wes_cfg = adapt_base_config(high_capacity_cfg, inpute_size_dict["wes"])
bulk_cfg = adapt_base_config(high_capacity_cfg, inpute_size_dict["bulk"])
sc_cfg = adapt_base_config(high_capacity_cfg, inpute_size_dict["scRNA"])

In [110]:
## Following latest update, we can specify the type of network so we need to change the config structure:
bulk_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": bulk_cfg}
hne_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": hne_cfg}
sc_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": sc_cfg}
wes_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": wes_cfg}
clinical_cfg = {"net_type": "mlp",
                    "device": device,
                    "net_config": clinical_cfg}

In [111]:
mme_cfg = {"hne_cfg":hne_cfg, 
           "clinical_cfg":clinical_cfg, 
           "wes_cfg":wes_cfg, 
           #"spatial_cfg":spatial_cfg,
           "bulk_cfg":bulk_cfg,
          "sc_cfg":sc_cfg}

## Pour instantier les classifieurs auxilliaires

In [112]:
set(patient_map.values())

{0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94}

In [113]:
aux_patient_cfg = {"layers": [out*len(mme_cfg),64,64,len(set(patient_map.values()))],
        "dropout": 0.4,
        "act_fn":[SmoothingFunction, SmoothingFunction, None],
        "norm_layer":norm_fn,
        "device":device}
aux_mod_cfg = {"layers": [out,32,32,len(mme_cfg)],
        "dropout": 0.4,
        "act_fn":[SmoothingFunction, SmoothingFunction, None],
        "norm_layer":norm_fn,
        "device":device}

In [114]:
def check_inf(tensor):
    return torch.isinf(tensor).sum() != 0
    
def check_nan(tensor):
    return torch.isnan(tensor).sum() != 0

def check(tensor):
    return check_inf(tensor) or check_nan(tensor)

def show_tensor(tensor, mode='grad'):
    plt.figure(figsize=(15,15))
    if mode == 'grad':
        if len(tensor.numpy().shape) == 1:
            plt.imshow(tensor.numpy().reshape(1,-1))
        else:
            plt.imshow(tensor.numpy())
        title = f"Average layer grad: {tensor.mean()}"
    else:
        if len(tensor.detach().numpy().shape) == 1:
            plt.imshow(tensor.detach().numpy().reshape(1,-1))
        else:
            plt.imshow(tensor.detach().numpy())
        title = f"Average layer output: {tensor.mean()}\nRatio of 0 : {(tensor == 0).sum() / tensor.size(0)}"
    plt.colorbar()
    plt.tight_layout()
    plt.title(title)
    plt.show()
    
def run_debug(EPOCHS, 
              temp, 
              sim, 
              alpha=0.1,
              bound=-10, 
              slope=0.05,
              rate=-2,
              auxilliary=False,
              aux_coef=0.5,
             ):
    EPOCHFLAG = 0
    problematic_batch_patient_ids = []
    # 1. Define model encoders + loss
    mme = MultiModalEncoder(**mme_cfg).to(device)
    # mme = torch.jit.script(mme)

    if auxilliary:
        auxilliary_patient = AuxilliaryClassifier(**aux_patient_cfg)
        loss_fn_aux_p = nn.CrossEntropyLoss()
        
        auxilliary_mod = AuxilliaryClassifier(**aux_mod_cfg)
        loss_fn_aux_m = nn.CrossEntropyLoss()

        aux_loss_smoother = SmoothingFunction(bound=bound,
                                         slope=slope,
                                         rate=rate)
    loss_fn = RegularizedInfoNCELoss(list(name_emb_dict.keys()),
                                     patient_map, 
                                     temperature=temp, 
                                     similarity=sim, 
                                     use_all_positives=False,
                                     alpha=alpha,
                                     bound=bound,
                                     slope=slope,
                                     rate=rate)
    param_dict = {"parameters":mme.parameters(), "lr":1e-3}
    if auxilliary:
        
        optimizer = Adam(
            [{"params":mme.parameters(), "lr":2e-3},
             {"params":auxilliary_patient.parameters(), "lr":2e-3},
             {"params":auxilliary_mod.parameters(), 'lr':2e-3}],
        )
        DATALOADER = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_patient_wise_colearning, generator=torch.Generator(device=dataset.device))
    else:
        optimizer = Adam(mme.parameters(),
                         lr=1e-3)
        DATALOADER = dataloader
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-4)
    # print(dict(mme.named_parameters()))
    # 2. Training loop
    EPOCH_LOSSES = []
    for epoch in range(EPOCHS):
        epoch_loss = []
        
        if auxilliary:
            epoch_cont_loss = []
            epoch_aux_m_loss = []
            epoch_aux_p_loss = []
        rankme = []
        for i, batch in enumerate(DATALOADER):
            # batch = (X_dict, patient_ids, available_modalities)
            if auxilliary:
                patient_ids, modalities, X_dict, avail_mods, aux_targets_p = batch
                aux_targets_p = torch.tensor(list(map(patient_map.get, aux_targets_p)), dtype=torch.long, device=device)
            else:
                patient_ids, modalities, X_dict, avail_mods = batch
            # print("patient_ids:", patient_ids)
            # print("modalities:", modalities)
            # print("X_dict:", X_dict)
            # print("avail_mods:", avail_mods)
        
            # print(X_dict)
            encoded = mme(X_dict)

            NANFLAG = {}
            for mod in X_dict.keys():
                if check(X_dict[mod]):
                    print("INPUT", X_dict[mod])
                    NANFLAG[mod] = 1
                    
                for key, tensor in dict(mme.wes_net.named_parameters()).items():
                    if check(tensor):
                        print("WEIGHTS", dict(mme.wes_net.named_parameters()))
                        NANFLAG[mod] = 1
                        break
                if check(encoded[mod]):
                    print("ENCODED", encoded[mod])
                    NANFLAG[mod] = 1
            if 1 in list(NANFLAG.values()):
                # print("Batch patient ids:", patient_ids)
                problematic_batch_patient_ids.append(patient_ids)
                EPOCHFLAG = 1
                break
                # raise ValueError(f"Found NANs in modality {[key for key in NANFLAG.keys() if NANFLAG[key] == 1]}")
            else:
                # # 2b. Pack into expected batch for loss
                loss_batch = (encoded, patient_ids, avail_mods)
                mme_embs = []
                if auxilliary:
                    input_aux_m = []
                    aux_targets_m = []
                for pid in range(encoded[list(encoded.keys())[0]].size(0)): 
                    patient_tensors = [encoded[mod][pid] for mod in encoded.keys()]
                    if auxilliary:
                        input_aux_m.extend([tensor.view(1,-1) for tensor in patient_tensors])
                        aux_targets_m.extend([i for i,mod in enumerate(encoded.keys())])
                    
                    mme_embs.append(torch.cat(patient_tensors, dim=0).unsqueeze(0))
                
                mme_embs = torch.cat(mme_embs, dim=0)
                if auxilliary:
                    input_aux_m = torch.cat(input_aux_m, dim=0)
                    aux_targets_m = torch.tensor(aux_targets_m, dtype=torch.long, device=device)

                    loss_aux_p = aux_loss_smoother(loss_fn_aux_p(auxilliary_patient(mme_embs),aux_targets_p))
                    loss_aux_m = aux_loss_smoother(loss_fn_aux_m(auxilliary_mod(input_aux_m),aux_targets_m))
            
                # # 2c. Compute loss & backward
                cont_loss = loss_fn(loss_batch) 
                loss = cont_loss if not auxilliary else cont_loss + aux_coef * (loss_aux_p - loss_aux_m)
                optimizer.zero_grad()
                # dot = make_dot(loss, params=dict(mme.named_parameters()), show_attrs=True, show_saved=True)
                # try:
                #     dot.view()
                # except:
                #     pass
                loss.backward()
                # for mod in X_dict.keys():
                #     param_dict = dict(mme.modality_net_map[mod].named_parameters())
                #     tensor = param_dict[list(param_dict.keys())[0]] # 1st layer parameters
                #     print(f"{key} GRADS:", tensor.grad)
                #     show_tensor(tensor.grad)
                #     print(f"Average 1st layer grad for {mod.upper()}: {tensor.mean()}")
                grads = []
                for param in mme.parameters():
                    gradlists = param.grad.tolist()
                    if isinstance(gradlists, list):
                        for gradlist in gradlists:
                            if isinstance(gradlist, list):
                                grads.extend(gradlist)
                            else:
                                grads.append(gradlist)
                    else:
                        grads.append(gradlists)
                print("MAX GRAD:", np.max(grads))
                print("AVG GRAD:", np.mean(grads))
                print("MIN GRAD:", np.min(grads))
                torch.nn.utils.clip_grad_norm_(mme.parameters(), max_norm=1.0)
                if auxilliary:
                    torch.nn.utils.clip_grad_norm_(auxilliary_patient.parameters(), max_norm=1.0)
                    torch.nn.utils.clip_grad_norm_(auxilliary_mod.parameters(), max_norm=1.0)
                optimizer.step()
                before_lr = optimizer.param_groups[0]["lr"]
                scheduler.step()
                after_lr = optimizer.param_groups[0]["lr"]
                epoch_loss.append(loss.item())
                if auxilliary:
                    epoch_cont_loss.append(cont_loss.item())
                    epoch_aux_p_loss.append(loss_aux_p.item())
                    epoch_aux_m_loss.append(loss_aux_m.item())
                
                # print(f"Batch {i} loss: {loss.item():.4f}")
        pos_align = np.mean(loss_fn.infonce.pos_alignments)
        neg_align = np.mean(loss_fn.infonce.neg_alignments)
        print(f"Epoch {epoch} total loss: {np.mean(epoch_loss):.4f}".upper())
        if auxilliary:
            print(f"\nEpoch {epoch} CONTRASTIVE loss: {np.mean(epoch_cont_loss):.4f}".upper())
            print(f"Epoch {epoch} AUXILLIARY losses: MODALITY CLF={np.mean(epoch_aux_m_loss):.4f}, PATIENT CLF={np.mean(epoch_aux_p_loss):.4f}".upper())
        print(f"\nEpoch {epoch} Embedding quality (Alignement): {pos_align:.4f}".upper())
        print(f"Epoch {epoch} Embedding quality (Negative Alignement): {neg_align:.4f}".upper())
        print(f"Epoch {epoch} Embedding quality (Alignement ratio): {np.abs(pos_align/(neg_align + 1e-8)):.4f}".upper())
        loss_fn.infonce.clear_alignments()
        if EPOCHFLAG == 1:
            print("BREAK")
            break
        EPOCH_LOSSES.append(np.mean(epoch_loss))
    return EPOCH_LOSSES, mme

In [115]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from umap import UMAP

from typing import Dict, List, Optional, Union, Tuple
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches


def visualize_embeddings(embeddings: Union[torch.Tensor, np.ndarray], 
                         method: str = 'pca',
                         labels: Optional[List] = None,
                         modality_info: Optional[Dict[str, int]] = None,
                         patient_info: Optional[Dict[str, int]] = None,
                         figsize: Tuple[int, int] = (12, 10),
                         title: Optional[str] = None,
                         show_legend: bool = True,
                         colors: Optional[List[str]] = None,
                         markers: Optional[List[str]] = None,
                         save_path: Optional[str] = None) -> None:
    """
    Visualize embeddings using dimensionality reduction techniques (PCA, t-SNE, or UMAP).
    
    Parameters:
    -----------
    embeddings : torch.Tensor or numpy.ndarray
        The embeddings to visualize. Shape should be [n_samples, n_features].
    method : str, default='pca'
        The dimensionality reduction method to use ('pca', 'tsne', or 'umap').
    labels : list, optional
        Labels for each embedding point. If None, no color-coding.
    modality_info : dict, optional
        Dictionary mapping modality names to indices for color-coding by modality.
    patient_info : dict, optional
        Dictionary mapping patient IDs to indices for grouping by patient.
    figsize : tuple, default=(12, 10)
        Figure size as (width, height) in inches.
    title : str, optional
        Title for the plot. If None, a default title based on the method is used.
    show_legend : bool, default=True
        Whether to show the legend.
    colors : list, optional
        List of colors to use for different categories. If None, defaults are used.
    markers : list, optional
        List of markers to use. If None, defaults are used.
    save_path : str, optional
        Path to save the figure. If None, the figure is displayed but not saved.
    
    Returns:
    --------
    None
        The function displays or saves the visualization.
    """
    # Convert torch tensor to numpy if needed
    if isinstance(embeddings, torch.Tensor):
        embeddings = embeddings.detach().cpu().numpy()
    
    # Input validation
    if embeddings.ndim != 2:
        raise ValueError(f"Expected 2D input, got shape {embeddings.shape}")
    
    n_samples, n_features = embeddings.shape
    
    # Choose dimensionality reduction method
    if method.lower() == 'pca':
        if title is None:
            title = f'PCA Visualization of {n_samples} Embeddings'
        reducer = PCA(n_components=2)
    elif method.lower() == 'tsne':
        if title is None:
            title = f't-SNE Visualization of {n_samples} Embeddings'
        reducer = TSNE(n_components=2, perplexity=min(30, n_samples-1), random_state=42)
    elif method.lower() == 'umap':
        if title is None:
            title = f'UMAP Visualization of {n_samples} Embeddings'
        min_dist = 0.1 if n_samples > 100 else 0.5
        reducer = UMAP(n_components=2, min_dist=min_dist, n_neighbors=min(15, n_samples-1), random_state=42)
    else:
        raise ValueError(f"Unknown method: {method}. Choose from 'pca', 'tsne', or 'umap'.")
    
    # Check for zero variance features that could cause problems
    feature_vars = np.var(embeddings, axis=0)
    if np.any(feature_vars < 1e-10):
        print(f"Warning: {np.sum(feature_vars < 1e-10)} features have near-zero variance.")
        # Remove zero variance features
        non_zero_idx = feature_vars >= 1e-10
        embeddings = embeddings[:, non_zero_idx]
    
    # Apply dimensionality reduction
    try:
        embeddings_2d = reducer.fit_transform(embeddings)
    except Exception as e:
        print(f"Error during dimensionality reduction: {e}")
        # Add a small amount of noise if we have issues
        embeddings = embeddings + np.random.normal(0, 1e-5, embeddings.shape)
        embeddings_2d = reducer.fit_transform(embeddings)
    
    # Prepare visualization
    plt.figure(figsize=figsize)
    
    # Set default colors and markers
    if colors is None:
        colors = list(mcolors.TABLEAU_COLORS) + list(mcolors.CSS4_COLORS)
    if markers is None:
        markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'H', '+', 'x', '|', '_']
    
    # Plot based on the available information
    if modality_info and patient_info:
        # Group by both modality and patient
        modalities = list(modality_info.keys())
        patients = list(patient_info.keys())
        
        # Create a scatter plot for each modality-patient combination
        for p_idx, patient in enumerate(patients):
            for m_idx, modality in enumerate(modalities):
                mask = (np.array(labels) == f"{patient}_{modality}")
                if np.any(mask):
                    plt.scatter(
                        embeddings_2d[mask, 0], 
                        embeddings_2d[mask, 1],
                        c=colors[m_idx % len(colors)], 
                        marker=markers[p_idx % len(markers)],
                        label=f"{patient} - {modality}" if np.sum(mask) > 0 else None,
                        alpha=0.7,
                        s=70)
    elif modality_info:
        emb_df = pd.DataFrame(embeddings_2d, columns=["UMAP1", "UMAP2"])
        emb_df = emb_df.join(pd.Series(modality_info).to_frame("modality"))
        print(emb_df.head())
        sns.scatterplot(emb_df, x="UMAP1", y="UMAP2", hue="modality")
        # # Group by modality only
        # modalities = list(modality_info.keys())
        # for m_idx, modality in enumerate(modalities):
        #     if labels:
        #         mask = np.array([label.endswith(modality) if isinstance(label, str) else False for label in labels])
        #     else:
        #         # Assume embeddings are ordered by modality if no labels
        #         start_idx = modality_info[modality]
        #         end_idx = modality_info[list(modality_info.keys())[m_idx+1]] if m_idx+1 < len(modalities) else n_samples
        #         mask = np.zeros(n_samples, dtype=bool)
        #         mask[start_idx:end_idx] = True
            
        #     plt.scatter(
        #         embeddings_2d[mask, 0], 
        #         embeddings_2d[mask, 1],
        #         c=colors[m_idx % len(colors)], 
        #         label=modality if np.sum(mask) > 0 else None,
        #         alpha=0.7,
        #         s=70
        #     )
    elif patient_info:
        # Group by patient only
        patients = list(patient_info.keys())
        for p_idx, patient in enumerate(patients):
            if labels:
                mask = np.array([label.startswith(patient) if isinstance(label, str) else False for label in labels])
            else:
                # Assume embeddings are ordered by patient if no labels
                start_idx = patient_info[patient]
                end_idx = patient_info[list(patient_info.keys())[p_idx+1]] if p_idx+1 < len(patients) else n_samples
                mask = np.zeros(n_samples, dtype=bool)
                mask[start_idx:end_idx] = True
            
            plt.scatter(
                embeddings_2d[mask, 0], 
                embeddings_2d[mask, 1],
                c=colors[p_idx % len(colors)], 
                label=patient if np.sum(mask) > 0 else None,
                alpha=0.7,
                s=70
            )
    elif labels is not None:
        # Use provided labels for coloring
        unique_labels = sorted(set(labels))
        for i, label in enumerate(unique_labels):
            mask = np.array(labels) == label
            plt.scatter(
                embeddings_2d[mask, 0], 
                embeddings_2d[mask, 1],
                c=colors[i % len(colors)], 
                label=label if np.sum(mask) > 0 else None,
                alpha=0.7,
                s=70
            )
    else:
        # No grouping information, plot all points with same style
        plt.scatter(
            embeddings_2d[:, 0], 
            embeddings_2d[:, 1],
            c=colors[0], 
            alpha=0.7,
            s=70
        )
    
    # Add metadata to the plot
    plt.title(title, fontsize=14)
    plt.xlabel(f'{method.upper()} Component 1', fontsize=12)
    plt.ylabel(f'{method.upper()} Component 2', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    
    # Compute and display explained variance for PCA
    if method.lower() == 'pca' and isinstance(reducer, PCA):
        explained_var = reducer.explained_variance_ratio_
        plt.xlabel(f'PC1 ({explained_var[0]:.2%} variance)', fontsize=12)
        plt.ylabel(f'PC2 ({explained_var[1]:.2%} variance)', fontsize=12)
        
        # Add text about total explained variance
        total_var = sum(explained_var)
        plt.figtext(0.5, 0.01, f'Total explained variance: {total_var:.2%}', 
                    ha='center', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
    
    # Add zero ratio information
    zero_ratio = (np.abs(embeddings) < 1e-6).sum() / embeddings.size
    plt.figtext(0.01, 0.01, f'Zero ratio: {zero_ratio:.2%}', 
                fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
    
    # Show legend if requested
    if show_legend and (modality_info or patient_info or labels is not None):
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    
    plt.tight_layout()
    
    # Save or display the figure
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    else:
        plt.show()

In [116]:
# problematic_patient_ids = []
# for run in range(10): 
#     print(f'RUN {run}')
#     problematic_patient_ids += run_debug(0.07, "original")

In [117]:
# problematic_patient_ids

In [118]:
# all_problematic_ids = []
# for id_list in problematic_patient_ids:
#     all_problematic_ids += id_list
# unique_ids = np.unique(all_problematic_ids)

# uid_presence = {uid:0 for uid in unique_ids}
# for uid in unique_ids:
#     for batch in problematic_patient_ids:
#         if uid in batch:
#             uid_presence[uid] += 1
# uid_presence

### Pas de patients commun à tous les batchs problématiques

### Les exponentielles montent forts => Gradient chelous: Apparemment courant pour la InfoNCELoss, il faut à tout prix tune le scaling parameter
To adress this, we need to find a good temperature. We also can try another similarity definition: cosine.

In [119]:
def show_weights(module: torch.nn.Module):
    """
    Plots histograms of all named parameter weights in the given PyTorch module.
    
    Args:
        module (nn.Module): The PyTorch module whose parameters will be visualized.
    """
    # Collect parameters
    params = list(module.named_parameters())
    num_params = len(params)
    if num_params == 0:
        print("Module has no parameters to show.")
        return
    
    # Determine subplot grid size
    cols = min(4, num_params)
    rows = (num_params + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows))
    axes = axes.flatten() if num_params > 1 else [axes]
    
    for idx, (name, param) in enumerate(params):
        data = param.detach().cpu().numpy().ravel()
        axes[idx].hist(data, bins=50)
        axes[idx].set_title(name)
        axes[idx].set_xlabel("Weight value")
        axes[idx].set_ylabel("Frequency")
    
    # Hide any unused subplots
    for j in range(idx+1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.show()
    
def log_and_plot_runs(RUNS, 
                      run_debug, 
                      EPOCHS, 
                      temp, 
                      sim, 
                      alpha=0.1,
                      bound=-10,
                      slope=0.05,
                      rate=-2,
                      auxilliary=False,
                      aux_coef=0.5,
                      show_w=False, 
                      show_out_all=True, 
                      show_emb=True):
    """
    Executes `run_debug` multiple times, logs final_loss per epoch for each run,
    and then plots all runs plus the average curve in bold.
    
    Args:
        RUNS (int): Number of runs to perform.
        EPOCHS (int): Number of epochs per run.
        run_debug (callable): Function with signature run_debug(EPOCHS, temp, sim, alpha=0.1, bound=-10, beta=0.2, nce_eps=1e-8, reg_eps=1e-8) 
                              returning a list/array of losses per epoch and the module.
        temp: Temperature parameter passed to run_debug.
        sim: Similarity parameter passed to run_debug.
    """
    # Store losses for all runs
    all_runs_losses = np.zeros((RUNS, EPOCHS))
    run_outputs = []
    for run in range(RUNS):
        print(f"\nRUN {run}:")
        losses, mme = run_debug(EPOCHS, 
                                temp, 
                                sim, 
                                alpha=alpha,
                                bound=bound,
                                slope=slope,
                                rate=rate,
                                auxilliary=auxilliary,
                                aux_coef=aux_coef
                                )  # expects length EPOCHS
        all_runs_losses[run:] = losses
        out_dict = mme(batch_all[2])
        if show_w:
            show_weights(mme)
        if show_out_all:
            for pid in range(5): # 5 premiers
                show_tensor(torch.cat([out_dict[mod][pid].view(-1) for mod in out_dict.keys()], dim=0), mode='out')
        
        mme_embs = []
        sep_mod_embs = []
        mod_info = {}
        patient_info = {}
        # print(out_dict[mod].size())
        idx = 0
        for pid in range(out_dict[list(out_dict.keys())[0]].size(0)): 
            patient_tensors = [out_dict[mod][pid] for mod in out_dict.keys()]
            mme_embs.append(torch.cat(patient_tensors, dim=0).unsqueeze(0))
        
            sep_mod_embs += [tensor.unsqueeze(0) for tensor in patient_tensors]
            for mod in out_dict.keys():
                mod_info[idx] = mod
                patient_info[idx] = pid
                idx += 1
        # print(mme_embs[0].size())
        mme_embs = torch.cat(mme_embs, dim=0)
        sep_mod_embs = torch.cat(sep_mod_embs, dim=0)
        # print(mod_info)
        # print(mme_embs.size())
        run_outputs.append([mme_embs,sep_mod_embs])
        if show_emb:
            visualize_embeddings(mme_embs, 'umap')
            visualize_embeddings(sep_mod_embs, 'umap', modality_info=mod_info)
            visualize_embeddings(sep_mod_embs, 'umap', modality_info=patient_info)
    # Compute average curve
    avg_curve = all_runs_losses.mean(axis=0)
    
    # Plotting
    plt.figure(figsize=(8, 5))
    
    # Plot each run
    for run in range(RUNS):
        plt.plot(range(1, EPOCHS+1), all_runs_losses[run], linewidth=1, c='grey', label=f'Run {run+1}')
    
    # Plot average in bold
    plt.plot(range(1, EPOCHS+1), avg_curve, linewidth=3, label='Average', zorder=10)
    
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss Curves for Each Run with Average')
    plt.legend()
    plt.grid(True)
    plt.show()
    return run_outputs

### Avec la version Cosine de la InfoNCELoss (appellée NT-Xent), après quelques essais, des bons paramètres sont:

In [120]:
run_outputs = log_and_plot_runs(1, run_debug, 50, 1, "nt-xent", 
                  alpha=0, # Coefficient de la régularisation inverse -> Favorise peu de 0 dans les emb de sorties
                  bound=-50, # Defini l'intervalle -BOUND, BOUND pour la loss, on utilisera tout le temps ça je pense
                  rate=-5,
                  slope=0.1,
                  auxilliary=False,
                  show_w=True, show_out_all=True, show_emb=True)


RUN 0:
Using device: cpu
Using device: cpu
Using device: cpu
Using device: cpu
Using device: cpu


/home/sagemaker-user/gbm_hackathon/gbmhackathon/utils/module_functions.py:26: ParameterNotFoundWarning: Parameter device not found in init arguments: ['self', 'layers', 'dropout', 'act_fn', 'norm_layer', 'enable_residuals'].
  warnings.warn(


tensor(5.8773, grad_fn=<DivBackward0>)
MAX GRAD: 0.011295784264802933
AVG GRAD: -2.2419646116999016e-06
MIN GRAD: -0.009554853662848473
tensor(5.8516, grad_fn=<DivBackward0>)
MAX GRAD: 0.008078619837760925
AVG GRAD: 1.0073355565402627e-06
MIN GRAD: -0.0067516048438847065
tensor(3.9175, grad_fn=<DivBackward0>)
MAX GRAD: 0.03203579783439636
AVG GRAD: 1.4720957903406212e-06
MIN GRAD: -0.03825892508029938
EPOCH 0 TOTAL LOSS: 12.4177

EPOCH 0 EMBEDDING QUALITY (ALIGNEMENT): 0.1311
EPOCH 0 EMBEDDING QUALITY (NEGATIVE ALIGNEMENT): 0.0067
EPOCH 0 EMBEDDING QUALITY (ALIGNEMENT RATIO): 19.6042
Alignments cleared
tensor(5.8290, grad_fn=<DivBackward0>)
MAX GRAD: 0.007356478367000818
AVG GRAD: 5.959507734515029e-07
MIN GRAD: -0.0065860147587955
tensor(5.8777, grad_fn=<DivBackward0>)
MAX GRAD: 0.006903413683176041
AVG GRAD: -7.616802777464793e-07
MIN GRAD: -0.0065832664258778095
tensor(3.9968, grad_fn=<DivBackward0>)
MAX GRAD: 0.027321849018335342
AVG GRAD: 7.0251406302579945e-06
MIN GRAD: -0.028619

KeyboardInterrupt: 

## Let's try using a co-learning framework with auxilliary classifiers
To further increase patient separability, we can try learning:
- A network that predicts patient (root id) => Goal: Increase classification accuracy => This should help push patients apart
- A network that predicts modalities for each modality output => Goal: Decrease classification accuracy => This should help bring patient modalities closer to each other furthermore

In [ ]:
run_outputs = log_and_plot_runs(1, run_debug, 50, 1, "nt-xent", 
                  alpha=1e-3, # Coefficient de la régularisation inverse -> Favorise peu de 0 dans les emb de sorties
                  bound=-50, # Defini l'intervalle -BOUND, BOUND pour la loss, on utilisera tout le temps ça je pense
                  rate=-5,
                  slope=0.1,
                  auxilliary=True,
                  aux_coef = 1e-3,
                  show_w=True, show_out_all=True, show_emb=True)

In [ ]:
BUCKET_MOSAIC = "ABSTRA_DATASET_03bb30aa_16ed_4b89_913e_fe009db2aabd"
BUCKET_PROJECT = "ABSTRA_PROJECT_STORAGE_BUCKET"

def fetch_path(env_var_name):
    return os.path.expandvars(f"${env_var_name}")

S3_PATH_CLINICAL_EMB = fetch_path(BUCKET_PROJECT) + "embedding_V1/2025-03-30_14-23_clinical_emb_V1.pkl"
S3_PATH_PROJECT = fetch_path(BUCKET_PROJECT)
clinical_dict = load_s3(S3_PATH_CLINICAL_EMB)

In [ ]:
id2row = clinical_dict['dataset']['id2row']
X = clinical_dict['dataset']['X']
Y = clinical_dict['dataset']['Y']
features = clinical_dict['dataset']['features']
targets = clinical_dict['dataset']['targets']
per_mod_contributions = clinical_dict['dataset']['mca_contributions']

In [ ]:
targets = targets[:2] + ["largest_diameter", targets[-1]]
Y_df = pd.DataFrame(Y.numpy(), columns=targets, index=list(id2row.keys()))

In [ ]:
patient_map_df = pd.Series(patient_map).to_frame("patient")
patient_map_df

In [ ]:
Y_df = Y_df.join(patient_map_df)
Y_df.head()

In [ ]:
if "patient" not in targets:
    targets.append("patient")
Y_df["patient"].astype(object)

In [ ]:
def analyze_run_outputs(run_results):
    umap = UMAP(random_state=42)  # Set random_state for reproducibility
    
    for run in range(len(run_results)):
        # Convert tensor to numpy array
        if torch.is_tensor(run_results[run]):
            MME_df = pd.DataFrame(run_results[run].detach().cpu().numpy(), 
                                  index=list(dataset.ind2patient.values()))
        else:
            MME_df = pd.DataFrame(run_results[run], 
                                  index=list(dataset.ind2patient.values()))
        
        # Apply UMAP dimensionality reduction
        umap_X = pd.DataFrame(umap.fit_transform(MME_df), 
                            columns=['UMAP1', 'UMAP2'], 
                            index=MME_df.index)
        
        # Combine with target variables
        umap_df = umap_X.join(Y_df)

        # Create figure with the correct number of subplots based on the number of targets
        num_targets = len(targets)
        n_rows = (num_targets + 1) // 2  # Calculate rows needed (ceiling division)
        n_cols = min(2, num_targets)     # Max 2 columns
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
        
        # Make axes iterable even when there's only one subplot
        if num_targets == 1:
            axes = np.array([axes])
        
        # Flatten axes array for easy indexing
        if num_targets > 1:
            axes = axes.flatten()
        
        # Create one subplot for each target
        for i, target in enumerate(targets):
            ax = axes[i] if num_targets > 1 else axes
            
            # Check if target column exists and is not all NaN
            if target in umap_df.columns and not umap_df[target].isna().all():
                # Make sure the target column is properly encoded for coloring
                if umap_df[target].dtype == 'object' or umap_df[target].dtype.name == 'category':
                    # For categorical data, make sure there's a proper mapping
                    scatter = sns.scatterplot(data=umap_df, x='UMAP1', y='UMAP2', hue=target, ax=ax)
                else:
                    # For numerical data, use a colormap
                    scatter = sns.scatterplot(data=umap_df, x='UMAP1', y='UMAP2', hue=target, 
                                             palette='inferno', ax=ax)
                
                # Only add legend if there are actually labeled points
                handles, labels = ax.get_legend_handles_labels()
                if len(handles) > 0:
                    ax.legend(handles=handles, labels=labels, bbox_to_anchor=(1.05, 1), 
                             loc='upper left', title=target)
                else:
                    print(f"Warning: No labels found for target '{target}'. Check if column exists or has valid values.")
            else:
                # If target doesn't exist or is all NaN, just plot the points without color encoding
                ax.scatter(umap_df['UMAP1'], umap_df['UMAP2'], alpha=0.5, color='gray')
                print(f"Warning: Target '{target}' not found in data or contains only NaN values.")
            
            # Set title
            ax.set_title(f"RUN: {run}, TARGET: {target}")
        
        # Remove any empty subplots
        if num_targets < n_rows * n_cols:
            for i in range(num_targets, n_rows * n_cols):
                if i < len(axes):  # Make sure we don't try to access non-existent axes
                    fig.delaxes(axes[i])
        
        # Adjust layout to prevent overlap
        plt.tight_layout()
        plt.show()

In [ ]:
analyze_run_outputs([run[0] for run in run_outputs])

In [ ]:
from itertools import permutations, product
original_mod = (0,1,1,0)
[t for t in list(product([0,1], repeat=4)) if sum(t) > 0 and t != original_mod]